# Section 3: AI Coding Tools + CLI Q&A Tool Development

> **Estimated time: 90–120 minutes**
> **Goal:** Set up Claude Code with DeepSeek and a `CLAUDE.md` project brief, then use it to build a command-line Q&A tool with paragraph citations.
> **Done when:** `cli_qa.py` runs — paste text, ask a question, get an answer with [Paragraph X] citations.
> **Prerequisites:** Completed [Section 2](Section2_API_LLM_Prompt.ipynb)

---

> **Note: Copy the commands below into a terminal and run them there. Do not run code directly in this notebook.**

## 3.1 Overview of AI-Assisted Programming

### What Is AI-Assisted Programming?

AI-assisted programming means using artificial intelligence to help plan, write, test, and revise code. In this workshop we will practice **two levels of control**:

### Mode 1: "AI advises you"

You ask Claude Code to explain a file, propose a small change, or show a code snippet, then you type or apply the change yourself. This keeps the learner in control.

**Typical scenarios:**
- Ask: `Explain how this function reads input until END.`
- Ask: `Show a minimal example of reading a CSV; do not edit files.`

### Mode 2: "AI writes for you"

You **describe the desired feature in natural language**, and the AI generates complete code directly. This approach has a popular name: **Vibe Coding** — you describe the "vibe" and the AI turns it into code.

**Typical scenarios:**
- You say "Write a function that reads a PDF file and extracts its text", and the AI generates the entire function
- You say "Create a FastAPI endpoint that takes a user question and returns an AI answer", and the AI generates the full API code

### In This Workshop...

We will experience **both modes**:
- First, use Claude Code + DeepSeek as an adviser: explain, plan, and review (Mode 1)
- Then, authorize Claude Code to create and revise files from a clear PRD (Mode 2)

---

## 3.2 Verify Claude Code + DeepSeek

Section 1 installed Claude Code and configured it to call DeepSeek directly through the Anthropic-compatible endpoint. Before editing the project, verify that setup from the project root.

### Step 1: Start Claude Code

```bash
cd smartlearn-agent
claude
```

Claude Code starts with the current directory as its project context. It can inspect files, propose changes, edit files, and run commands after you approve them.

### Step 2: Verify the model

Inside Claude Code, run:

```text
/model
```

Confirm that the model is `deepseek-v4-pro` (or the DeepSeek model configured in Section 1). Then ask:

```text
List the files in this project. Do not modify anything.
```

> **Important boundary:** Claude Code calls DeepSeek through `https://api.deepseek.com/anthropic`. OpenRouter remains separate and is used by the Python exercises to teach API calls.

✅ **Checkpoint 1:** `/model` shows DeepSeek and Claude Code can list the project files.

> **Troubleshooting:**
> - `claude: command not found` → run `npm install -g @anthropic-ai/claude-code`, then reopen the terminal
> - `/model` does not show DeepSeek → recheck `ANTHROPIC_BASE_URL`, `ANTHROPIC_AUTH_TOKEN`, and `ANTHROPIC_MODEL` from Section 1
> - Authentication fails → confirm the token is a DeepSeek Platform key, not an OpenRouter key

---

## 3.3 Create the `CLAUDE.md` Project Brief

Claude Code automatically reads `CLAUDE.md` from the project root. Treat it as a durable briefing: project purpose, stack, commands, safety rules, and the boundary between the coding assistant and the learning API.

### Steps

1. In the project root, create a file named `CLAUDE.md`.
2. Copy the template below into it.
3. Replace the two `TODO` items with concrete project information.
4. Save the file, then ask Claude Code: `Read CLAUDE.md and summarize the rules you must follow. Do not modify files.`

> `CLAUDE.md` may be committed because it contains project guidance, not secrets. Never put API keys in it.


```
# SmartLearn Agent

## Project
TODO: Describe the project in one sentence

## Tech Stack
- Backend: Python + FastAPI
- Frontend: React + Vite
- LLM: OpenRouter (qwen/qwen3.5-flash-02-23)
- Vector Search: FAISS (Day 3)

## AI Coding Environment
- Claude Code uses DeepSeek directly through ANTHROPIC_BASE_URL
- OpenRouter is only for the student Python API exercises
- Never route Claude Code through OpenRouter

## Conventions
- API keys in .env, never commit
- Use venv for Python dependencies
- Commit messages: type: description (feat/fix/docs/refactor)

## Do Not Modify
TODO: List files that should not be changed
```


**Template guide:**

| Section | Description | Example |
|------|------|------|
| `## Project` | Describe the project in one sentence | `An AI-powered learning assistant that answers course-related questions` |
| `## Tech Stack` | Technologies used in the project (pre-filled) | Keep as-is |
| `## Conventions` | Team coding conventions (pre-filled) | Keep as-is |
| `## Do Not Modify` | Files that should never be changed | `- .env`, `- package-lock.json` |

#### Step 3: Fill in the TODO sections

Replace both `TODO` markers with concrete content. For example:

```markdown
## Project
SmartLearn Agent is an AI-powered learning assistant that parses PDF lecture slides and answers students' course-related questions.

## Do Not Modify
- .env
- package-lock.json
```

#### Step 4: Save the file

- **Mac:** `Cmd + S`
- **Windows:** `Ctrl + S`

✅ **Checkpoint 2:** The project root contains `CLAUDE.md`, both `TODO` parts are replaced, and Claude Code can summarize its rules.

---

## 3.4 Commit `CLAUDE.md` to Git

### Purpose

`CLAUDE.md` is part of the project and should be managed in Git. When teammates clone the repository, Claude Code automatically picks up the same project context.

### Steps

In the terminal, make sure the current directory is the project root (the `smartlearn-agent` folder), then run the following commands in order:

In [ ]:
# Step 1: Stage CLAUDE.md
# git add CLAUDE.md

> **Expected output:** No output at all (silent success).

In [ ]:
# Step 2: Commit to the local repository
# git commit -m "docs: add Claude Code project context"

> **Expected output:**
> ```
> [main xxxxxxx] docs: add Claude Code project context
>  1 file changed, XX insertions(+)
>  create mode 100644 CLAUDE.md
> ```

Git has recorded this change. `xxxxxxx` is the commit's unique ID (different every time).

In [ ]:
# Step 3: Push to the GitHub remote repository
# git push

> **Expected output:**
> ```
> Enumerating objects: 4, done.
> Counting objects: 100% (4/4), done.
> ...
> To github.com:your-username/smartlearn-agent.git
>    xxxxxxx..xxxxxxx  main -> main
> ```

The code has been uploaded to GitHub.

✅ **Checkpoint 3:** `CLAUDE.md` is committed and pushed to GitHub. Open the GitHub repository page to confirm the file appears.

> **Troubleshooting:**
> - `git add` fails with `fatal: not a git repository` → The current directory is outside the project folder; run `cd smartlearn-agent` to switch into it
> - `git push` fails with `Authentication failed` → GitHub authentication needs to be set up; see the GitHub authentication steps in Section 1.13
> - `git push` fails with `rejected` → The remote repository may have newer commits; run `git pull` to sync, then `git push` again

---

## 3.5 Hands-On: Ask Claude Code About the Project

### Purpose

With Claude Code and `CLAUDE.md` configured, verify that the assistant can read the project and distinguish the two API paths.

### Steps

#### Step 1: Start Claude Code from the project root

Run `claude`, then use `/model` to confirm the DeepSeek model.

#### Step 2: Ask a read-only question

Type the following question into the chat box:

```
What files are in this project?
```

Press `Enter` to send.

> **Expected output:**
>
> Claude Code scans the current project directory and should mention files like these:
>
> ```
> Here are the files in your project:
> - CLAUDE.md
> - .gitignore
> - README.md
> - .env (if it exists)
> ...
> ```

![placeholder: Claude Code terminal listing the project files]

#### Step 3 (optional): Try one more question

Follow up with:

```
Based on CLAUDE.md, explain which API Claude Code uses and which API the Python exercises use. Do not modify files.
```

> **Expected output:**
>
> Claude Code should report: the coding assistant calls DeepSeek directly through the Anthropic-compatible endpoint, while the Python exercises call OpenRouter to learn API usage.

✅ **Checkpoint 4:** Claude Code identifies the project files and explains the DeepSeek/OpenRouter boundary from `CLAUDE.md`.

> **Troubleshooting:**
> - The file list is wrong → exit and run `claude` again from inside `smartlearn-agent`
> - The model is wrong → recheck the DeepSeek environment variables from Section 1
> - Claude Code ignores the rules → confirm the file is named exactly `CLAUDE.md` and is in the project root

---

---

# Part 2: Build the CLI Q&A Tool

> The AI coding tools are ready. Now for today's core hands-on project.

## 3.6 Project Goal (5 minutes)

### The End Result

We will write a Python command-line tool, `cli_qa.py`, that works like this:

1. The user pastes multi-paragraph text (typing `END` to finish the input)
2. The user types a question
3. The program calls an LLM, answers the question based on the text, and labels which paragraph the answer came from (e.g. `[Paragraph 1]`)

### Demo: Expected Input and Output

```
$ python3 cli_qa.py

请粘贴文本（输入 END 结束）：
Python was created by Guido van Rossum and first released in 1991.
It was designed to be easy to read.

Python uses indentation instead of curly braces to define code blocks.
This makes Python code visually clean and consistent.

Python has a large standard library.
END

请输入你的问题：Who created Python?

回答：
Python was created by Guido van Rossum and was first released in 1991 [Paragraph 1].
```

### Core Concept: RAG

The core idea behind this tool is **RAG (Retrieval-Augmented Generation)**. Enterprise Q&A bots, document search tools, customer support systems — these AI applications all share the same underlying pattern:

1. **Give the LLM reference material** (here, the text the user pastes in)
2. **Have the LLM answer questions based on that material** (with citations, so users can verify where answers come from)

Today the reference material is pasted in by hand. In real products, this step is replaced by automatically retrieving relevant content from a database or search engine — the principle is identical.

![placeholder: A flow diagram showing user input text → split and number paragraphs → assemble prompt → call LLM → return answer with citations]

---

## 3.7 Write a PRD (Product Requirements Document) (10 minutes)

### What Is a PRD?

A PRD (Product Requirements Document) is a short document that describes **what to build**. Clarifying requirements before writing code is standard practice in professional development.

**When generating code with AI, the PRD is the instruction you give the AI.** The clearer the description, the more accurate the generated code. A vague PRD forces the AI to guess, and the generated code will likely miss expectations.

### Hands-On: Write `cli_qa_prd.md` Before Any Code

Create `cli_qa_prd.md` in the project root. Write requirements and tests before code. Claude Code will read this file in the next exercise.

In [ ]:
# This block is not meant to run. Save the content as cli_qa_prd.md.

"""
CLI Q&A Tool - PRD (Product Requirements Document)

What it does:
  A command-line tool that takes a multi-paragraph text and a question,
  then uses an LLM to answer the question with paragraph-level citations.

Input:
  1. Multi-line text from user (terminated by typing 'END' on a new line)
  2. A question about the text

Output:
  An answer that references specific paragraphs using [Paragraph X] format.

Done when / acceptance tests:
  - User can paste text and ask questions in the terminal
  - Answers include [Paragraph X] citations
  - Uses OpenRouter API (google/gemma-4-31b-it:free model)
  - API key loaded from .env file and never printed
  - A question answered by Paragraph 1 cites [Paragraph 1]
  - A question absent from the text returns: The text does not provide this information.
  - An empty text input shows a friendly error instead of calling the API
  - python3 -m py_compile cli_qa.py succeeds
"""

### Notes on the PRD

- **What it does** summarizes the feature in one or two sentences
- **Input** specifies the input format and where it comes from
- **Output** specifies what the output looks like
- **Done when** lists observable acceptance tests — commands, inputs, outputs, and failure behavior

A useful PRD lets another person decide pass/fail without asking what you meant. If `Done when` only says 'it works', rewrite it with a concrete input and expected output. Five minutes spent here prevents the coding agent from inventing product scope.

> **The same applies to prompts for AI:** the more specific the instruction to Claude Code (what input, what output, which libraries, which API), the more reliable the generated code. Vague instructions = vague results.

---

## 3.8 Decompose the Problem (10 minutes)

### Break the Big Problem into Small Steps

Before writing code, list out what the program needs to do. In programming this is called **decomposition** — splitting a complex problem into a series of simple steps.

Our `cli_qa.py` needs to do these 7 things:

| Step | What | Why |
|------|--------|--------|
| 1 | Read multi-line text from the user (until `END` is typed) | The user needs a way to paste in the reference material |
| 2 | Split the text into paragraphs by blank lines | The LLM needs to know which paragraph is which to cite accurately |
| 3 | Number each paragraph: `[Paragraph 1]`, `[Paragraph 2]`... | Numbering lets the LLM cite specific paragraphs in its answer |
| 4 | Read the user's question | Know what the user wants to ask |
| 5 | Build the prompt: system message + numbered text + question | The LLM needs explicit instructions to answer in the required format |
| 6 | Call the LLM API | Send the prompt to the AI model and get the answer |
| 7 | Print the answer | Show the result to the user |

### One Function per Step

In programming, each independent step usually becomes a **function**. A function is like a small machine: it takes some input, processes it, and produces output.

```
read_text()          → Step 1: read the text
split_paragraphs()   → Step 2: split into paragraphs
number_paragraphs()  → Step 3: number them
input()              → Step 4: read the question (Python built-in)
ask_question()       → Steps 5 + 6: build the prompt and call the API
print()              → Step 7: print the result (Python built-in)
main()               → tie all the steps together
```

This "decompose first, implement step by step" approach is the fundamental method every programmer uses.

---

## 3.9 Teacher-Led Claude Code Loop (30 minutes)

### Round 0: Establish a Safe Checkpoint

Before starting Claude Code, run `git status`. Commit or set aside your own work so the new diff is easy to inspect. Then start `claude` and run `/model` to confirm DeepSeek.

### Round 1: Experience an Underspecified Request (Plan Only)

Send this first; do **not** let it edit files:

```text
I need a command-line question-answering tool. Propose a plan only. Do not edit files.
```

Write down every assumption Claude Code makes: input format, API, model, citations, errors, and files. This is your baseline.

### Round 2: Give It the PRD and Demand a Testable Plan

Now send:

```
Read CLAUDE.md and cli_qa_prd.md. Do not edit files yet.
Return exactly:
1. Requirements you understood
2. Files you expect to create or modify
3. Implementation steps
4. Acceptance tests with exact commands or inputs
5. Ambiguities or risks
```

Compare Round 1 and Round 2. The better plan should be narrower and easier to test, not merely longer.

### Round 3: Implement the Approved Slice

After reviewing the plan, send:

```text
Implement the approved plan. Do not read, print, or modify .env.
Keep the change limited to cli_qa.py and any small test fixture you truly need.
After editing, run python3 -m py_compile cli_qa.py.
Then show: changed files, test result, and any remaining risk.
```

![placeholder: Claude Code terminal showing plan, focused diff, and test result]

### Review the Generated Code

Do not judge success by whether a file appeared. Success means the diff matches the PRD and the acceptance checks pass. Review every changed file before continuing.

### Four Questions to Ask While Reading AI-Generated Code

**Question 1: How does it read input until `END`?**

Find the part that reads user input and see what logic decides "the user is done typing". It is usually a `while` loop that reads one line at a time and stops when it sees `END`.

In [ ]:
# Example: the typical pattern for reading input until END
# Compare with the code Claude Code generated and check whether it uses similar logic

lines = []
while True:
    line = input()           # Read one line
    if line.strip() == "END":  # If the line is END (after stripping whitespace)
        break                # Stop the loop
    lines.append(line)       # Otherwise add the line to the list

**Question 2: How does it split paragraphs?**

Paragraphs are separated by blank lines. The code needs to group consecutive non-empty lines into one paragraph and start a new one at each blank line. A common approach is to split on `"\n\n"`.

In [ ]:
# Example: split paragraphs by blank lines
text = """First paragraph line 1.
First paragraph line 2.

Second paragraph line 1.

Third paragraph."""

# Split on two consecutive newline characters
paragraphs = text.split("\n\n")

# Filter out empty paragraphs (multiple consecutive blank lines can produce empty strings)
paragraphs = [p.strip() for p in paragraphs if p.strip()]

for i, p in enumerate(paragraphs, 1):
    print(f"--- Paragraph {i} ---")
    print(p)
    print()

> **Expected output:**
> ```
> --- Paragraph 1 ---
> First paragraph line 1.
> First paragraph line 2.
>
> --- Paragraph 2 ---
> Second paragraph line 1.
>
> --- Paragraph 3 ---
> Third paragraph.
> ```

**Question 3: What does the system prompt say?**

The system prompt is the "role instruction" for the LLM — it tells the model how to answer. Ours needs two key requirements:
- Answer only from the provided text (no making things up)
- Cite source paragraphs in the `[Paragraph X]` format

**Question 4: What happens if the text is very long?**

Every LLM has a **context window** limit — the maximum amount of text it can process at once. If the text exceeds the limit, the API returns an error. Today's exercise skips this concern, but real-world development must account for it.

### Make Sure Dependencies Are Installed

Whatever code Claude Code generated, two Python libraries are required:
- `openai`: for calling the LLM API (OpenRouter is compatible with the OpenAI interface format)
- `python-dotenv`: for reading the API key from the `.env` file

Run in the terminal:

In [ ]:
pip3 install openai python-dotenv

> **Expected output:**
> ```
> Successfully installed openai-1.x.x python-dotenv-1.x.x
> ```
> If they are already installed, `Requirement already satisfied` appears instead — that is also fine.

### Make Sure the .env File Exists

The project directory should already contain a `.env` file (created in Section 2). Confirm it contains the OpenRouter API key:

In [ ]:
cat .env

> **Expected output:**
> ```
> OPENROUTER_API_KEY=YOUR_OPENROUTER_API_KEY
> ```

If the file does not exist or the line is missing, create it:
```bash
echo "OPENROUTER_API_KEY=your-api-key" > .env
```

✅ **Checkpoint 1:** `openai` and `python-dotenv` are installed, and `.env` contains `OPENROUTER_API_KEY`.

---

## 3.10 Understand the Code Line by Line (15 minutes)

Below is the complete reference implementation. Whatever version Claude Code generated, read this code carefully — every line has a comment explaining what it does.

If the Claude Code-generated version runs correctly, keep using it. If it has problems, replace it with this reference implementation.

### Reference Implementation: Complete cli_qa.py

In [ ]:
# ============================================================
# cli_qa.py - CLI Q&A Tool with Paragraph Citations
# ============================================================
# This script lets the user paste a passage of text, ask a question,
# and calls an LLM to generate an answer with paragraph citations.
# ============================================================

import os                          # OS utilities (used here to read environment variables)
from openai import OpenAI          # OpenAI SDK, used to call the LLM API
from dotenv import load_dotenv     # Load environment variables from the .env file

# ---- Load environment variables ----
# load_dotenv() reads the .env file in the current directory
# and sets its key-value pairs as environment variables.
# This keeps the API key out of the source code.
load_dotenv()


def read_text():
    """
    Step 1: Read multi-line text from the user.
    The user types or pastes text line by line, then types END to finish.
    Returns: one complete string (all lines joined with newlines).
    """
    print("请粘贴文本（输入 END 结束）：")
    lines = []                     # Collect each line in a list
    while True:                    # Loop until we hit break
        line = input()             # Read one line of user input
        if line.strip() == "END":  # Compare after .strip() removes surrounding whitespace
            break                  # Exit the loop
        lines.append(line)         # Otherwise add the line to the list
    return "\n".join(lines)        # Join all lines into one string with newlines


def split_paragraphs(text):
    """
    Step 2: Split the text into paragraphs by blank lines.
    Two consecutive newline characters (\n\n) mark a paragraph boundary.
    Returns: a list where each element is one paragraph's text.
    """
    # split("\n\n") splits on double newlines
    # strip() removes surrounding whitespace from each paragraph
    # if p.strip() filters out empty paragraphs
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    return paragraphs


def number_paragraphs(paragraphs):
    """
    Step 3: Add a number to each paragraph.
    Input: ['text of paragraph A', 'text of paragraph B', ...]
    Output: one formatted string, with [Paragraph 1] etc. before each paragraph.
    """
    numbered = []
    for i, para in enumerate(paragraphs, 1):   # enumerate starts numbering at 1
        numbered.append(f"[Paragraph {i}]\n{para}")
    return "\n\n".join(numbered)   # Separate paragraphs with a blank line


def ask_question(numbered_text, question):
    """
    Steps 5 + 6: Build the prompt and call the LLM API.
    - numbered_text: the numbered reference text
    - question: the user's question
    Returns: the LLM's answer (a string).
    """
    # ---- Create the API client ----
    # Use the OpenAI SDK, but point base_url at OpenRouter.
    # This lets us call many different models through OpenRouter.
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),  # Read the API key from environment variables
    )

    # ---- system prompt ----
    # The role instruction for the LLM: how it should answer
    system_prompt = (
        "You are a helpful assistant. "
        "Answer the user's question based ONLY on the provided text. "
        "Cite your sources by referencing [Paragraph X] where X is the paragraph number. "
        "If the answer is not in the text, say so."
    )

    # ---- user prompt ----
    # Combine the numbered text and the user's question into one message
    user_prompt = f"""Here is the text:

{numbered_text}

Question: {question}"""

    # ---- Call the API ----
    response = client.chat.completions.create(
        model="google/gemma-4-31b-it:free",  # Use the free model
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    # response.choices[0].message.content is the text returned by the LLM
    return response.choices[0].message.content


def main():
    """
    Main function: tie all the steps together.
    """
    # Step 1: read the text
    text = read_text()

    # Step 2: split into paragraphs
    paragraphs = split_paragraphs(text)
    print(f"\n检测到 {len(paragraphs)} 个段落。\n")

    # Step 3: number the paragraphs
    numbered_text = number_paragraphs(paragraphs)

    # Step 4: read the question
    question = input("请输入你的问题：")

    # Steps 5 + 6: call the LLM
    print("\n正在思考...\n")
    answer = ask_question(numbered_text, question)

    # Step 7: print the answer
    print("回答：")
    print(answer)


# ---- Entry point ----
# Run main() only when this file is executed directly (rather than imported by another file)
if __name__ == "__main__":
    main()

### Function-by-Function Walkthrough

#### `read_text()` — read user input

This function uses a `while True` infinite loop to keep reading lines of user input. When the user types `END`, the `break` statement exits the loop. All lines collected so far are stored in the `lines` list, and `"\n".join(lines)` joins them into one complete string at the end.

Key detail: the `.strip()` in `line.strip() == "END"` removes surrounding whitespace, so END is still recognized even if the user accidentally adds spaces around it.

#### `split_paragraphs()` — split on blank lines

One line of code does three things (this style is called a **list comprehension** in Python):
```python
paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
```
1. `text.split("\n\n")` — split the text on double newlines
2. `p.strip()` — remove surrounding whitespace from each paragraph
3. `if p.strip()` — keep only non-empty paragraphs

#### `number_paragraphs()` — number the paragraphs

`enumerate(paragraphs, 1)` iterates over the list while providing numbers starting at 1. `f"[Paragraph {i}]\n{para}"` is Python's **f-string** formatting syntax; `{i}` and `{para}` are replaced with the actual values.

#### `ask_question()` — build the prompt and call the API

This is the most important function. It does three things:
1. Creates the OpenAI client (pointed at OpenRouter)
2. Builds two messages: `system` (role instruction) and `user` (numbered text + question)
3. Calls `client.chat.completions.create()` to send the request

#### `main()` — tie everything together

`main()` calls the functions above in order, forming the complete workflow. `if __name__ == "__main__"` is a standard Python idiom meaning "run `main()` only when this file is executed directly".

✅ **Checkpoint 2:** Understood what each function does — `read_text` reads input, `split_paragraphs` splits paragraphs, `number_paragraphs` numbers them, `ask_question` calls the API, and `main` connects the flow.

---

## 3.11 Test with Sample Text (15 minutes)

### Prepare the Test Text

Copy the passage about Python below; it will be pasted into the program in a moment. It has 5 paragraphs, separated by blank lines:

```
Python was created by Guido van Rossum and was first released in 1991. It was named after the British comedy group Monty Python. Guido wanted a language that was fun to use and easy to read.

Python uses indentation to define code blocks instead of curly braces or keywords. This makes Python code visually clean and forces developers to write readable code. Most other languages use curly braces { } for this purpose.

Python supports multiple programming paradigms including procedural, object-oriented, and functional programming. This flexibility allows developers to choose the best approach for their specific problem.

The Python Package Index (PyPI) hosts over 400,000 third-party packages. Popular packages include NumPy for scientific computing, Django for web development, and TensorFlow for machine learning.

Major companies that use Python include Google, Netflix, Instagram, and Spotify. Google uses Python for many internal tools, and YouTube's backend was originally written in Python.
```

### Run the Program

Make sure the current directory is the project directory, then run:

In [ ]:
python3 cli_qa.py

After the program starts:
1. When the prompt `请粘贴文本（输入 END 结束）：` appears, paste the test text above
2. After pasting, type `END` on a new line and press Enter
3. The message `检测到 5 个段落。` appears
4. When `请输入你的问题：` appears, type a question

![placeholder: Terminal running cli_qa.py, showing "检测到 5 个段落" after pasting the text]

### Test Five Behaviors

Run the program once per question and complete all five checks:

#### Test 1: Who created Python?

**Expected behavior:**
- The answer should mention Guido van Rossum
- It should cite `[Paragraph 1]`, since the creator information is in the first paragraph

> **Expected output (example):**
> ```
> 回答：
> Python was created by Guido van Rossum and was first released in 1991 [Paragraph 1].
> He named it after the British comedy group Monty Python [Paragraph 1].
> ```

#### Test 2: What makes Python different in terms of syntax?

**Expected behavior:**
- The answer should mention indentation instead of curly braces
- It should cite `[Paragraph 2]`

> **Expected output (example):**
> ```
> 回答：
> Python uses indentation to define code blocks instead of curly braces or keywords
> [Paragraph 2]. This design choice makes Python code visually clean and forces
> developers to write readable code [Paragraph 2].
> ```

#### Test 3: What companies use Python?

**Expected behavior:**
- The answer should mention Google, Netflix, Instagram, and Spotify
- It should cite `[Paragraph 5]`

> **Expected output (example):**
> ```
> 回答：
> Major companies that use Python include Google, Netflix, Instagram, and Spotify
> [Paragraph 5]. Google uses Python for many internal tools, and YouTube's backend
> was originally written in Python [Paragraph 5].
> ```

#### Test 4: Which database does Python use?

**Expected behavior:**
- The answer must not invent a database
- It should return the exact not-found sentence defined in `cli_qa_prd.md`

This fourth test is essential: a document Q&A tool is not reliable unless it knows when to refuse.

#### Test 5: Empty input

Start the program and immediately type `END`. It should show a friendly error and must not call OpenRouter.

![placeholder: Terminal evidence for cited answer, unsupported question refusal, and empty-input handling]

### Debugging Citation Accuracy

The LLM sometimes skips citations or cites the wrong paragraph. This usually means the system prompt lacks specificity.

**The fix: improve the system prompt.** Modify `system_prompt` in the `ask_question()` function to make the instructions more explicit.

**Before:**
```python
system_prompt = (
    "You are a helpful assistant. "
    "Answer the user's question based ONLY on the provided text. "
    "Cite your sources by referencing [Paragraph X] where X is the paragraph number. "
    "If the answer is not in the text, say so."
)
```

**After (a more explicit version):**
```python
system_prompt = (
    "You are a precise research assistant. "
    "Rules:\n"
    "1. Answer ONLY using information from the provided text.\n"
    "2. After EVERY claim, add a citation in the format [Paragraph X].\n"
    "3. If a sentence uses information from multiple paragraphs, cite all of them.\n"
    "4. If the text does not contain the answer, reply: "
    "'The provided text does not contain information to answer this question.'\n"
    "5. Do NOT add any information beyond what is in the text."
)
```

This is the essence of **prompt engineering** — improving output quality by adjusting the instructions given to the LLM. In real development, tuning the prompt is often the most direct way to improve results.

### When a Test Fails: Use an Evidence-Rich Debug Prompt

Do not send only `It does not work`. Paste this template into Claude Code and fill every field:

```text
Command I ran:
<exact command>

Full traceback or incorrect output:
<paste everything>

Expected behavior:
<one observable result>

Actual behavior:
<what happened>

Recent change:
<what changed immediately before the failure>

Diagnose from this evidence. Propose the smallest fix only; no unrelated refactor.
Before editing, tell me the root cause and the exact command you will rerun.
```

After Claude Code fixes it, rerun the failing case **and** one previously passing case. This checks both the fix and regression risk.

✅ **Checkpoint 3:** The program passed cited-answer, unsupported-question, and empty-input tests. Save the terminal evidence or write the observed result next to each acceptance test.

> **Troubleshooting:**
> - `ModuleNotFoundError: No module named 'openai'` → Run `pip3 install openai python-dotenv`
> - `openai.AuthenticationError` or `401 Unauthorized` → Check that `OPENROUTER_API_KEY` in the `.env` file is correct
> - `openai.APIConnectionError` → Check the network connection and confirm `openrouter.ai` is reachable
> - The program hangs at "正在思考..." for a long time → The free model can be slow to respond; wait about 30 seconds. If there is no response after 1 minute, press `Ctrl + C` and try again
> - Answers contain no citations → Improve the system prompt (see the improved version above)

---

## 3.12 Iterate and Improve (20 minutes)

The program runs — now let's make it better. This section has three exercises that make the tool more practical.

### Challenge 1: Citations Could Be More Accurate — Improve the System Prompt

If the LLM sometimes skips citations or cites the wrong paragraph, the system prompt can be tuned further.

**Strategy: show the LLM an example (few-shot prompting)**

Add an example to the system prompt so the LLM knows the expected answer format:

In [ ]:
# In the ask_question() function, replace system_prompt with the following:

system_prompt = """You are a precise research assistant.

Rules:
1. Answer ONLY using information from the provided text.
2. After EVERY claim, add a citation in the format [Paragraph X].
3. If a sentence uses information from multiple paragraphs, cite all of them.
4. If the text does not contain the answer, reply:
   "The text does not provide this information."
5. Do NOT add any information beyond what is in the text.

Example:
If the text says:
[Paragraph 1] The sky is blue.
[Paragraph 2] Grass is green.

And the question is: 'What color is the sky?'
Your answer should be: 'The sky is blue [Paragraph 1].'
"""

Rerun the program after the change and test with the same text and questions. The citations should be noticeably more accurate.

This technique of providing examples inside the prompt is called **few-shot prompting** — one of the most practical techniques in prompt engineering.

---

### Challenge 2: Read from a File — Add a `--file` Flag

Pasting text by hand every time is tedious. Let's support reading directly from a file:
```bash
python3 cli_qa.py --file sample.txt
```

This uses Python's `argparse` module — the standard library dedicated to handling command-line arguments.

#### What Are Command-Line Arguments?

When `python3 cli_qa.py --file sample.txt` is typed in the terminal, `--file sample.txt` is a **command-line argument**. Programs can read these arguments to change their behavior.

`argparse` is built into Python (no extra installation needed) and exists specifically to define and parse these arguments.

In [ ]:
# Add the import at the top of cli_qa.py
import argparse

# Then modify the main() function:

def main():
    # ---- Parse command-line arguments ----
    parser = argparse.ArgumentParser(description="CLI Q&A Tool")
    parser.add_argument(
        "--file",
        help="Path to a text file to use as input (instead of pasting)",
    )
    args = parser.parse_args()

    # ---- Read the text ----
    if args.file:
        # If --file was given, read from the file
        with open(args.file, "r", encoding="utf-8") as f:
            text = f.read()
        print(f"已从文件 {args.file} 读取文本。")
    else:
        # Otherwise read from the keyboard
        text = read_text()

    # ---- The rest of the flow is unchanged ----
    paragraphs = split_paragraphs(text)
    print(f"\n检测到 {len(paragraphs)} 个段落。\n")

    numbered_text = number_paragraphs(paragraphs)

    question = input("请输入你的问题：")

    print("\n正在思考...\n")
    answer = ask_question(numbered_text, question)

    print("回答：")
    print(answer)

#### Create the Test File

Save the test text as a file first:

In [ ]:
cat > sample.txt << 'EOF'
Python was created by Guido van Rossum and was first released in 1991. It was named after the British comedy group Monty Python. Guido wanted a language that was fun to use and easy to read.

Python uses indentation to define code blocks instead of curly braces or keywords. This makes Python code visually clean and forces developers to write readable code. Most other languages use curly braces { } for this purpose.

Python supports multiple programming paradigms including procedural, object-oriented, and functional programming. This flexibility allows developers to choose the best approach for their specific problem.

The Python Package Index (PyPI) hosts over 400,000 third-party packages. Popular packages include NumPy for scientific computing, Django for web development, and TensorFlow for machine learning.

Major companies that use Python include Google, Netflix, Instagram, and Spotify. Google uses Python for many internal tools, and YouTube's backend was originally written in Python.
EOF

#### Test File Input Mode

In [ ]:
python3 cli_qa.py --file sample.txt

> **Expected output:**
> ```
> 已从文件 sample.txt 读取文本。
>
> 检测到 5 个段落。
>
> 请输入你的问题：
> ```

The program read the text directly from the file, skipping the manual paste step. Type a question to get an answer.

✅ **Checkpoint 4:** `python3 cli_qa.py --file sample.txt` reads text directly from the file and the Q&A works normally.

---

### Challenge 3: Multi-Turn Q&A — Add a Loop Mode

Currently the program answers one question and exits. Let's add a loop so the user can ask multiple questions about the same text, typing `quit` to exit.

In [ ]:
# Modify the Q&A part of main() to add a while loop:

def main():
    # ---- Parse command-line arguments ----
    parser = argparse.ArgumentParser(description="CLI Q&A Tool")
    parser.add_argument(
        "--file",
        help="Path to a text file to use as input (instead of pasting)",
    )
    args = parser.parse_args()

    # ---- Read the text ----
    if args.file:
        with open(args.file, "r", encoding="utf-8") as f:
            text = f.read()
        print(f"已从文件 {args.file} 读取文本。")
    else:
        text = read_text()

    paragraphs = split_paragraphs(text)
    print(f"\n检测到 {len(paragraphs)} 个段落。")

    numbered_text = number_paragraphs(paragraphs)

    # ---- Q&A loop ----
    print("你可以连续提问，输入 quit 退出。\n")

    while True:
        question = input("请输入你的问题（quit 退出）：")

        if question.strip().lower() == "quit":
            print("再见！")
            break

        print("\n正在思考...\n")
        answer = ask_question(numbered_text, question)

        print("回答：")
        print(answer)
        print()  # Blank line to visually separate each round

#### Test Multi-Turn Q&A

In [ ]:
python3 cli_qa.py --file sample.txt

> **Expected output:**
> ```
> 已从文件 sample.txt 读取文本。
>
> 检测到 5 个段落。
> 你可以连续提问，输入 quit 退出。
>
> 请输入你的问题（quit 退出）：Who created Python?
>
> 正在思考...
>
> 回答：
> Python was created by Guido van Rossum [Paragraph 1].
>
> 请输入你的问题（quit 退出）：What companies use Python?
>
> 正在思考...
>
> 回答：
> Major companies that use Python include Google, Netflix, Instagram,
> and Spotify [Paragraph 5].
>
> 请输入你的问题（quit 退出）：quit
> 再见！
> ```

Multiple questions can now be asked about the same text!

✅ **Checkpoint 5:** Multi-turn mode works — questions can be asked back to back, and typing `quit` exits.

> **Troubleshooting:**
> - Typing `quit` does nothing → Confirm `.strip().lower()` handles case and whitespace; try typing exactly `quit` with nothing else
> - The second question raises an error → Check whether `ask_question()` creates a fresh client on each call (in the code above, the client is created inside the function, so every call creates a new one)

---

### Final Complete Version

The complete `cli_qa.py` with all the improvements integrated:

In [ ]:
"""
CLI Q&A Tool - PRD (Product Requirements Document)

What it does:
  A command-line tool that takes a multi-paragraph text and a question,
  then uses an LLM to answer the question with paragraph-level citations.

Input:
  1. Multi-line text from user (terminated by typing 'END' on a new line)
     OR a text file via --file flag
  2. One or more questions about the text

Output:
  Answers that reference specific paragraphs using [Paragraph X] format.

Done when:
  - User can paste text or load from file
  - Answers include [Paragraph X] citations
  - Supports multiple questions in one session
  - Uses OpenRouter API (google/gemma-4-31b-it:free model)
  - API key loaded from .env file and never printed
  - Empty text exits with a friendly error before any API call
  - Missing answers return exactly: The text does not provide this information.
"""

import os
import argparse
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

MISSING = "The text does not provide this information."


def read_text():
    """Read multi-line text from the terminal until END is typed."""
    print("请粘贴文本（输入 END 结束）：")
    lines = []
    while True:
        line = input()
        if line.strip() == "END":
            break
        lines.append(line)
    return "\n".join(lines)


def split_paragraphs(text):
    """Split the text into a list of paragraphs by blank lines."""
    return [p.strip() for p in text.split("\n\n") if p.strip()]


def number_paragraphs(paragraphs):
    """Number the paragraphs and return one formatted string."""
    numbered = []
    for i, para in enumerate(paragraphs, 1):
        numbered.append(f"[Paragraph {i}]\n{para}")
    return "\n\n".join(numbered)


def ask_question(numbered_text, question):
    """Build the prompt, call the LLM API, and return the answer."""
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
        raise SystemExit("OPENROUTER_API_KEY is missing. Add it to .env and try again.")

    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
    )

    system_prompt = """You are a precise research assistant.

Rules:
1. Answer ONLY using information from the provided text.
2. After EVERY claim, add a citation in the format [Paragraph X].
3. If a sentence uses information from multiple paragraphs, cite all of them.
4. If the text does not contain the answer, reply exactly:
   'The text does not provide this information.'
5. Do NOT add any information beyond what is in the text.

Example:
If the text says:
[Paragraph 1] The sky is blue.
[Paragraph 2] Grass is green.

And the question is: 'What color is the sky?'
Your answer should be: 'The sky is blue [Paragraph 1].'
"""

    user_prompt = f"""Here is the text:

{numbered_text}

Question: {question}"""

    response = client.chat.completions.create(
        model="google/gemma-4-31b-it:free",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    return response.choices[0].message.content


def main():
    parser = argparse.ArgumentParser(description="CLI Q&A Tool")
    parser.add_argument(
        "--file",
        help="Path to a text file to use as input (instead of pasting)",
    )
    args = parser.parse_args()

    # Read the text
    if args.file:
        with open(args.file, "r", encoding="utf-8") as f:
            text = f.read()
        print(f"已从文件 {args.file} 读取文本。")
    else:
        text = read_text()

    paragraphs = split_paragraphs(text)
    if not paragraphs:
        raise SystemExit("No text was provided. Paste text or choose a non-empty file.")
    print(f"\n检测到 {len(paragraphs)} 个段落。")

    numbered_text = number_paragraphs(paragraphs)

    # Q&A loop
    print("你可以连续提问，输入 quit 退出。\n")

    while True:
        question = input("请输入你的问题（quit 退出）：")

        if question.strip().lower() == "quit":
            print("再见！")
            break

        print("\n正在思考...\n")
        answer = ask_question(numbered_text, question)

        print("回答：")
        print(answer)
        print()


if __name__ == "__main__":
    main()

✅ **Checkpoint 6:** Completed all three improvements — a more precise system prompt, file input mode, and multi-turn Q&A. The final version is fully integrated.

---

## 3.13 Commit the Code

Development is done — time to save the work to Git.

### Check the Changes

In [ ]:
git status

> **Expected output:**
> ```
> Untracked files:
>   (use "git add <file>..." to include in what will be committed)
>         cli_qa.py
>         sample.txt
> ```

### Stage and Commit

In [ ]:
git add cli_qa.py cli_qa_prd.md
git commit -m "feat: CLI Q&A with paragraph citation"

> **Expected output:**
> ```
> [main xxxxxxx] feat: CLI Q&A with paragraph citation
>  1 file changed, XX insertions(+)
>  create mode 100644 cli_qa.py
> ```

### Push to GitHub

In [ ]:
git push

> **Expected output:**
> ```
> Enumerating objects: 4, done.
> ...
> To github.com:yourname/your-repo.git
>    xxxxxxx..xxxxxxx  main -> main
> ```

✅ **Checkpoint 7:** `cli_qa.py` is committed and pushed to GitHub.

> **Troubleshooting:**
> - `git push` fails with an authentication error → Check that GitHub authentication is configured (see Section 1.13)
> - Accidentally staged `.env` → The `.env` file contains the API key and should never be uploaded to GitHub. Run `git restore --staged .env` to unstage it, then confirm `.gitignore` contains a `.env` line

---

## Section 3 Complete!

This section covered:

| Step | Content | Skill Learned |
|------|------|------------|
| 3.6 | Define the goal | Basic concepts of RAG |
| 3.7 | Write a PRD | Define requirements before writing code |
| 3.8 | Decompose the problem | Break a complex task into small steps |
| 3.9 | Generate code with AI | Write precise Claude Code prompts and review the resulting diff |
| 3.10 | Understand the code line by line | Read and understand Python functions and API calls |
| 3.11 | Test | Verify program behavior with sample data |
| 3.12 | Iterate and improve | Prompt engineering, argparse, loop mode |
| 3.13 | Commit to Git | Save and push code |

The core architecture of this CLI Q&A tool — **provide reference text + have the LLM answer from that text + cite sources** — is exactly how enterprise-grade RAG systems work. Replace "manually paste text" with "retrieve relevant documents from a database" and you have a complete AI knowledge-base product.

→ Next: [Section 4: Vibe Coding + Product Design](Section4_Vibe_Coding_Design.ipynb)